# Data Explpoite

In [6]:
import sys
import os
from pathlib import Path
# Remonte d'un niveau pour atteindre la racine (parent de /notebook)
sys.path.insert(0, os.path.abspath(".."))

In [11]:
from src.data.schema import TRANSACTIONS_SCHEMA
from pyspark.sql import SparkSession
from src.config import SparkConfig

cfg = SparkConfig()
spark = (
        SparkSession.builder
            .appName(cfg.app_name)
            .master(cfg.master)
            .config("spark.sql.shuffle.partitions", str(cfg.shuffle_partitions))
            .config("spark.sql.adaptive.enabled", str(cfg.adaptive_enabled).lower())
            .config("spark.sql.adaptive.coalescePartitions.enabled",
                    str(cfg.adaptive_enabled).lower())
            .config("spark.driver.memory", cfg.driver_memory)
            .config("spark.executor.memory", cfg.executor_memory)
            .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

In [12]:
path = Path().resolve().parent / "data" / "raw" / "online_retail_II.csv"

df = (
    spark.read
         .schema(TRANSACTIONS_SCHEMA)
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("nullValue", "")
         .option("mode", "DROPMALFORMED")
         .csv(str(path))
)

In [13]:
df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00|     6.95|     13085|United Kingdom|
|   489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00|     6.75|     13085|United Kingdom|
|   489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00|     6.75|     13085|United Kingdom|
|   489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|      2.1|     13085|United Kingdom|
|   489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00|     1.25|     13085|United Kingdom|
|   489434|    22064|PINK DOUGHNUT TRI...|      24|2009-12-01 07:45:00|     1.65|     13085|United Kingdom|
|   489434|    21871| SAVE T

In [17]:
from pyspark.sql import DataFrame, functions as F

df = (
    df.filter(F.col('CustomerID').isNotNull())
      .filter(F.col("Quantity") > 0)
      .filter(F.col("UnitPrice") > 0)
      .filter(~F.col("InvoiceNo").startswith("C"))
      .withColumn("Revenue", F.col("Quantity") * F.col("UnitPrice"))
      .dropDuplicates()
    )

In [28]:
total = df.count()
nulls = (
        df.select([
            F.sum(F.col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ]).collect()[0].asDict()
    )

In [29]:
nulls

{'InvoiceNo': 0,
 'StockCode': 0,
 'Description': 0,
 'Quantity': 0,
 'InvoiceDate': 0,
 'UnitPrice': 0,
 'CustomerID': 0,
 'Country': 0,
 'Revenue': 0}

In [30]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)



In [31]:
n_rows = df.count()
n_cust = df.select("CustomerID").distinct().count()
n_inv  = df.select("InvoiceNo").distinct().count()
n_sku  = df.select("StockCode").distinct().count()

In [32]:
print(f"Lignes        : {n_rows:>12,}")
print(f"Clients       : {n_cust:>12,}")
print(f"Factures      : {n_inv:>12,}")
print(f"Produits      : {n_sku:>12,}")

Lignes        :      779,425
Clients       :        5,878
Factures      :       36,969
Produits      :        4,631


In [35]:
df.agg(
    F.min("InvoiceDate").alias("date_min"),
    F.max("InvoiceDate").alias("date_max"),
).show(truncate=False)


+-------------------+-------------------+
|date_min           |date_max           |
+-------------------+-------------------+
|2009-12-01 07:45:00|2011-12-09 12:50:00|
+-------------------+-------------------+



In [36]:
(df.groupBy("Country")
    .agg(F.round(F.sum("Revenue"), 2).alias("revenu_total"),
        F.countDistinct("CustomerID").alias("nb_clients"))
    .orderBy(F.col("revenu_total").desc())
    .show(5, truncate=False))

+--------------+-------------+----------+
|Country       |revenu_total |nb_clients|
+--------------+-------------+----------+
|United Kingdom|1.438923492E7|5350      |
|EIRE          |616570.54    |5         |
|Netherlands   |554038.09    |22        |
|Germany       |425019.71    |107       |
|France        |348768.96    |95        |
+--------------+-------------+----------+
only showing top 5 rows


In [38]:
(df.groupBy("StockCode", "Description")
    .agg(F.sum("Quantity").alias("qte_totale"),
        F.round(F.sum("Revenue"), 2).alias("revenu_total"))
    .orderBy(F.col("qte_totale").desc())
    .show(5, truncate=False))

df.select("Quantity", "UnitPrice", "Revenue").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()


+---------+----------------------------------+----------+------------+
|StockCode|Description                       |qte_totale|revenu_total|
+---------+----------------------------------+----------+------------+
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS |105185    |24098.03    |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|91757     |247048.01   |
|23843    |PAPER CRAFT , LITTLE BIRDIE       |80995     |168469.6    |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |78234     |124351.86   |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |77916     |81416.73    |
+---------+----------------------------------+----------+------------+
only showing top 5 rows
+-------+------------------+------------------+------------------+
|summary|          Quantity|         UnitPrice|           Revenue|
+-------+------------------+------------------+------------------+
|  count|            779425|            779425|            779425|
|   mean|13.489369727683869|3.2184879853755906|22.291823161943626|
| 